# **FundusGuard AI**

model p3 using Efficient-Net U-Net

**Secure Kaggle key**

In [ ]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# **Collect Dataset**

In [ ]:
!kaggle datasets download -d arnavjain1/glaucoma-datasets

!unzip glaucoma-datasets.zip -d glaucoma_dataset

**File structure:**

+ glaucoma_dataset
  + G1020
    + Images
      - image_0.jpg
      - image_0.json
      - image_1.jpg
      - image_1.json
      
      .....
    + Images_Cropped
    + Image_Square
    + Masks
    + Masks_Cropped
    + Mask_Square
    + NerveRemoved_Images
    - G1020.CSV
  + ORIGA
    + Images
        - image_0.jpg
        - image_0.json
        - image_1.jpg
        - image_1.json
        
        .....
    + Images_Cropped
    + Image_Square
    + Masks
    + Masks_Cropped
    + Mask_Square
    + Semi-automatic-annotations
    - OrigaList.CSV
  + REFUGE
  + MODELS
    - refug_clf.pkl
    - refuge_segmentation.pth



# **Preprocessing Time**

In [ ]:
!pip install segmentation-models-pytorch torchinfo thop

In [ ]:
import os
import numpy as np
import cv2
from tensorflow.keras.preprocessing.image import ImageDataGenerator, array_to_img, img_to_array
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
from torchinfo import summary
from thop import profile

In [ ]:
def remove_nerves(image):
    img = array_to_img(image)
    
    img = cv2.cvtColor(np.array(img), cv2.COLOR_BGR2RGB)
    # convert image to grayScale
    grayScale = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
   
    # kernel for morphologyEx
    kernel = cv2.getStructuringElement(1,(17,17))
   
    # apply MORPH_BLACKHAT to grayScale image
    blackhat = cv2.morphologyEx(grayScale, cv2.MORPH_BLACKHAT, kernel)
  
    # apply thresholding to blackhat
    _,threshold = cv2.threshold(blackhat,10,255,cv2.THRESH_BINARY)

    # inpaint with original image and threshold image
    final_image = cv2.inpaint(img,threshold,1,cv2.INPAINT_TELEA)
    final_image = cv2.cvtColor(final_image, cv2.COLOR_BGR2RGB)
    
    return final_image.astype(np.float64)/255.0

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import Dataset, DataLoader,random_split

from scipy.ndimage.measurements import label
import matplotlib.pyplot as plt

In [ ]:
class GlaucomaDataset(Dataset):

    def __init__(self, root_dir, split='train', output_size=(256,256)):
        self.output_size = output_size
        self.root_dir = root_dir
        self.split = split
        self.images = []
        self.segs = []
        # Load data index
        for direct in self.root_dir:
            self.image_filenames = []
            for path in os.listdir(os.path.join(direct, "Images_Square")):
                if(not path.startswith('.')):
                    self.image_filenames.append(path)


            for k in range(len(self.image_filenames)):
                print('Loading {} image {}/{}...'.format(split, k, len(self.image_filenames)), end='\r')
                img_name = os.path.join(direct, "Images_Square", self.image_filenames[k])
                img = remove_nerves(np.array(Image.open(img_name).convert('RGB'))).astype(np.float32)
                img = np.array(Image.open(img_name).convert('RGB'))
                img = transforms.functional.to_tensor(img)
                img = transforms.functional.resize(img, output_size, interpolation=Image.BILINEAR)
                self.images.append(img)
            if split != 'test':
                for k in range(len(self.image_filenames)):
                    print('Loading {} segmentation {}/{}...'.format(split, k, len(self.image_filenames)), end='\r')
                    seg_name = os.path.join(direct, "Masks_Square", self.image_filenames[k][:-3] + "png")
                    mask = np.array(Image.open(seg_name, mode='r'))
                    od = (mask==1.).astype(np.float32)
                    oc = (mask==2.).astype(np.float32)
                    od = torch.from_numpy(od[None,:,:])
                    oc = torch.from_numpy(oc[None,:,:])
                    od = transforms.functional.resize(od, output_size, interpolation=Image.NEAREST)
                    oc = transforms.functional.resize(oc, output_size, interpolation=Image.NEAREST)
                    self.segs.append(torch.cat([od, oc], dim=0))

            print('Succesfully loaded {} dataset.'.format(split) + ' '*50)
            
            
    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        if self.split == 'test':
            return img
        else:
            seg = self.segs[idx]
            return img, seg

In [ ]:
EPS = 1e-7

def dice_coef_sample(input, target,smooth=1e-6):
    iflat = input.contiguous().view(-1)
    tflat = target.contiguous().view(-1)
    intersection = (iflat * tflat).sum()
    return (2. * intersection+ smooth) / (iflat.sum() + tflat.sum()+ smooth)

def compute_dice_coef(input, target):
    '''
    Compute dice score metric.
    '''
    batch_size = input.shape[0]
    return sum([dice_coef_sample(input[k,:,:], target[k,:,:]) for k in range(batch_size)])/batch_size


def vertical_diameter(binary_segmentation):
    '''
    Get the vertical diameter from a binary segmentation.
    The vertical diameter is defined as the "fattest" area of the binary_segmentation parameter.
    '''

    # get the sum of the pixels in the vertical axis
    vertical_axis_diameter = np.sum(binary_segmentation, axis=1)

    # pick the maximum value
    diameter = np.max(vertical_axis_diameter, axis=1)

    # return it
    return diameter


def vertical_cup_to_disc_ratio(od, oc):
    '''
    Compute the vertical cup-to-disc ratio from a given labelling map.
    '''
    # compute the cup diameter
    cup_diameter = vertical_diameter(oc)
    # compute the disc diameter
    disc_diameter = vertical_diameter(od)

    return cup_diameter / (disc_diameter + EPS)

def compute_vCDR_error(pred_od, pred_oc, gt_od, gt_oc):
    '''
    Compute vCDR prediction error, along with predicted vCDR and ground truth vCDR.
    '''
    pred_vCDR = vertical_cup_to_disc_ratio(pred_od, pred_oc)
    gt_vCDR = vertical_cup_to_disc_ratio(gt_od, gt_oc)
    vCDR_err = np.mean(np.abs(gt_vCDR - pred_vCDR))
    return vCDR_err, pred_vCDR, gt_vCDR


def classif_eval(classif_preds, classif_gts):
    '''
    Compute AUC classification score.
    '''
    auc = roc_auc_score(classif_gts, classif_preds)
    return auc


In [ ]:
def refine_seg(pred):
    '''
    Only retain the biggest connected component of a segmentation map.
    '''
    np_pred = pred.numpy()
        
    largest_ccs = []
    for i in range(np_pred.shape[0]):
        labeled, ncomponents = label(np_pred[i,:,:])
        bincounts = np.bincount(labeled.flat)[1:]
        if len(bincounts) == 0:
            largest_cc = labeled == 0
        else:
            largest_cc = labeled == np.argmax(bincounts)+1
        largest_cc = torch.tensor(largest_cc, dtype=torch.float32)
        largest_ccs.append(largest_cc)
    largest_ccs = torch.stack(largest_ccs)
    
    return largest_ccs

# **Data Spliting into Root, Test**

In [ ]:
root_dirs = [ "glaucoma_datasets/REFUGE","glaucoma_datasets/G1020"]
test_dir = [ "glaucoma_datasets/ORIGA"]
lr = 1e-4
batch_size = 8
num_workers = 8
total_epoch = 50
patience = 10

In [ ]:
full_train_dataset = GlaucomaDataset(root_dirs, split='train')

# data splite (80% Train, 20% Val)
total_size = len(full_train_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

train_set, val_set = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_set, 
                          batch_size=batch_size, 
                          shuffle=True, 
                          num_workers=num_workers,
                          pin_memory=True)

val_loader = DataLoader(val_set, 
                        batch_size=batch_size, 
                        shuffle=False,
                        num_workers=num_workers,
                        pin_memory=True)

test_set = GlaucomaDataset(test_dir)
test_loader = DataLoader(test_set, 
                         batch_size=batch_size, 
                         shuffle=False, 
                         num_workers=num_workers,
                         pin_memory=True)

# **Define the U-Net Architecture**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b3",      
    encoder_weights="imagenet",          
    in_channels=3,                       
    classes=2,                           
    activation='sigmoid'                
).to(device)


In [ ]:
seg_loss = torch.nn.BCELoss(reduction='mean')

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
summary(model, input_size=(1, 3, 256,256))

In [ ]:
inputs = torch.randn(1, 3, 224, 224).to(device)
macs, params = profile(model, inputs=(inputs, ))

print(f"Total MACs (Multiply-Accumulates): {macs}")
print(f"Total Parameters: {params}")

print(f"Estimasi FLOPs: {macs * 2}")
inputs = torch.randn(1, 3, 256,256).to(device)
macs, params = profile(model, inputs=(inputs, ))

print(f"Total MACs (Multiply-Accumulates): {macs}")
print(f"Total Parameters: {params}")

print(f"Estimasi FLOPs: {macs * 2}")

In [ ]:
best_val_auc = 0.
train_losses, val_losses = [], []
train_dsc_od_history, val_dsc_od_history = [], []
train_dsc_oc_history, val_dsc_oc_history = [], []


# **Training Phase**

In [ ]:
print("Starting Training...")
total_start_time = time.time()
counter = 0
for epoch in range(total_epoch):
    epoch_start_time = time.time()
    
    # Accumulators per Epoch
    train_loss, train_dsc_od, train_dsc_oc = 0., 0., 0.
    val_loss, val_dsc_od, val_dsc_oc = 0., 0., 0.
    train_vCDR_error, val_vCDR_error = 0., 0.
    
    model.train()
    print(f'\nEpoch {epoch + 1}/{total_epoch}')
    
    for batch_idx, (imgs, seg_gts) in enumerate(train_loader):
        imgs, seg_gts = imgs.to(device), seg_gts.to(device)

        # Forward pass
        logits = model(imgs)
        loss = seg_loss(logits, seg_gts)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
        with torch.no_grad():
            pred_od_raw = (logits[:, 0, :, :] >= 0.5).type(torch.int8)
            pred_oc_raw = (logits[:, 1, :, :] >= 0.5).type(torch.int8)
            
            gt_od = seg_gts[:, 0, :, :].type(torch.int8)
            gt_oc = seg_gts[:, 1, :, :].type(torch.int8)

            train_dsc_od += compute_dice_coef(pred_od_raw, gt_od).item()
            train_dsc_oc += compute_dice_coef(pred_oc_raw, gt_oc).item()

        print(f'   Train Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}', end='\r')

    avg_train_loss = train_loss / len(train_loader)
    avg_train_dsc_od = train_dsc_od / len(train_loader)
    avg_train_dsc_oc = train_dsc_oc / len(train_loader)

    model.eval()
    with torch.no_grad():
        for batch_idx, (imgs, seg_gts) in enumerate(val_loader):
            imgs, seg_gts = imgs.to(device), seg_gts.to(device)
            
            logits = model(imgs)
            val_loss += seg_loss(logits, seg_gts).item()

            pred_od_cpu = (logits[:, 0, :, :] >= 0.5).type(torch.int8).cpu()
            pred_oc_cpu = (logits[:, 1, :, :] >= 0.5).type(torch.int8).cpu()
            
            pred_od = refine_seg(pred_od_cpu).to(device)
            pred_oc = refine_seg(pred_oc_cpu).to(device)
            
            gt_od = seg_gts[:, 0, :, :].type(torch.int8)
            gt_oc = seg_gts[:, 1, :, :].type(torch.int8)

            val_dsc_od += compute_dice_coef(pred_od, gt_od).item()
            val_dsc_oc += compute_dice_coef(pred_oc, gt_oc).item()

            v_err, _, _ = compute_vCDR_error(
                pred_od.cpu().numpy(), pred_oc.cpu().numpy(), 
                gt_od.cpu().numpy(), gt_oc.cpu().numpy()
            )
            val_vCDR_error += v_err
            
            print(f'   Val Batch {batch_idx+1}/{len(val_loader)}', end='\r')

    avg_val_loss = val_loss / len(val_loader)
    avg_val_dsc_od = val_dsc_od / len(val_loader)
    avg_val_dsc_oc = val_dsc_oc / len(val_loader)
    avg_val_vcdr_err = val_vCDR_error / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_dsc_od_history.append(avg_train_dsc_od)
    val_dsc_od_history.append(avg_val_dsc_od)
    train_dsc_oc_history.append(avg_train_dsc_oc)
    val_dsc_oc_history.append(avg_val_dsc_oc)    

    # --- Print Summary ---
    epoch_duration = time.time() - epoch_start_time
    total_elapsed = time.time() - total_start_time
    
    print('\n' + '='*50)
    print(f'SUMMARY EPOCH {epoch + 1} | Time: {epoch_duration:.2f}s')
    print(f'Loss    : {avg_train_loss:.4f} (train) | {avg_val_loss:.4f} (val)')
    print(f'Dice OD : {avg_train_dsc_od:.4f} (train) | {avg_val_dsc_od:.4f} (val)')
    print(f'Dice OC : {avg_train_dsc_oc:.4f} (train) | {avg_val_dsc_oc:.4f} (val)')
    print(f'vCDR Err: {avg_val_vcdr_err:.4f} (val)')
    print(f'Total Elapsed: {total_elapsed/60:.2f} min')

    # Save Best Model
    current_val_score = avg_val_dsc_od + avg_val_dsc_oc
    if current_val_score > best_val_auc:
        best_val_auc = current_val_score
        model_name = 'gla_p3_'+best_val_auc+'.pth'
        torch.save(model.state_dict(), model_name)
        print('>>> Best validation Score reached. Model saved!')
        counter = 0
    #else:
    #    counter += 1
    #    print(f'>>> EarlyStopping counter: {counter} out of {patience}')
        
    #    if counter >= patience:
    #        print(f"!!! Early stopping. Training berhenti di Epoch {epoch + 1} !!!")
    #       break # Keluar dari loop training
    print('='*50)

# **Performance Graph**

In [ ]:
def plot_training_history(train_od, val_od, train_oc, val_oc):
    t_od = [0] + list(train_od)
    v_od = [0] + list(val_od)
    t_oc = [0] + list(train_oc)
    v_oc = [0] + list(val_oc)
    
    epochs = range(0, len(t_od)) 
    
    plt.figure(figsize=(15, 5))

    # Plot Dice OD
    plt.subplot(1, 2, 1)
    plt.plot(epochs, t_od, 'g', label='Train Dice OD')
    plt.plot(epochs, v_od, 'darkgreen', linestyle='--', label='Val Dice OD')
    plt.title('Optic Disc Dice Score')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.xlim(0, len(t_od) - 1)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # Plot Dice OC
    plt.subplot(1, 2, 2)
    plt.plot(epochs, t_oc, 'orange', label='Train Dice OC')
    plt.plot(epochs, v_oc, 'darkorange', linestyle='--', label='Val Dice OC')
    plt.title('Optic Cup Dice Score')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.xlim(0, len(t_oc) - 1)
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

plot_training_history(
    train_dsc_od_history, val_dsc_od_history, 
    train_dsc_oc_history, val_dsc_oc_history
)

In [ ]:
def plot_training_history(train_losses, val_losses, train_dice, val_dice, train_iou, val_iou):
    epochs_loss = range(1, len(train_losses) + 1)
    
    t_dice = [0] + list(train_dice)
    v_dice = [0] + list(val_dice)
    t_iou = [0] + list(train_iou)
    v_iou = [0] + list(val_iou)
    
    epochs_metrics = range(0, len(t_dice))
    
    plt.figure(figsize=(18, 5))

    plt.subplot(1, 3, 1)
    plt.plot(epochs_loss, train_losses, 'r', label='Train Loss')
    plt.plot(epochs_loss, val_losses, 'b', linestyle='--', label='Val Loss')
    plt.title('Training & Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.xlim(1, len(train_losses)) # Mulai dari 1
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs_metrics, t_dice, 'g', label='Train Dice')
    plt.plot(epochs_metrics, v_dice, 'darkgreen', linestyle='--', label='Val Dice')
    plt.title('Mean Dice Score (OD/OC)')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.xlim(0, len(t_dice) - 1)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs_metrics, t_iou, 'darkorange', label='Train IoU')
    plt.plot(epochs_metrics, v_iou, 'chocolate', linestyle='--', label='Val IoU')
    plt.title('Mean IoU (Jaccard Index)')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.xlim(0, len(t_iou) - 1)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend()

    plt.tight_layout()
    plt.show()

train_dice_mean = [(od + oc) / 2 for od, oc in zip(train_dsc_od_history, train_dsc_oc_history)]
val_dice_mean = [(od + oc) / 2 for od, oc in zip(val_dsc_od_history, val_dsc_oc_history)]

train_iou_mean = [d / (2 - d) if d < 1 else 1.0 for d in train_dice_mean]
val_iou_mean = [d / (2 - d) if d < 1 else 1.0 for d in val_dice_mean]

plot_training_history(
    train_losses, val_losses, 
    train_dice_mean, val_dice_mean,
    train_iou_mean, val_iou_mean
)

In [ ]:
def compute_all_metrics(pred, gt, eps=1e-7):
    pred = pred.view(-1).bool()
    gt = gt.view(-1).bool()

    tp = (pred & gt).sum().item()
    fp = (pred & ~gt).sum().item()
    fn = (~pred & gt).sum().item()
    tn = (~pred & ~gt).sum().item()

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * (precision * recall) / (precision + recall + eps)
    iou = tp / (tp + fp + fn + eps)

    return iou, precision, recall, f1

In [ ]:
def find_best_thresholds_separate(model, loader, device):
    model.eval()
    thresholds = np.arange(0.1, 0.95, 0.05)
    
    best_t_od = 0.5
    best_t_oc = 0.5
    best_dice_od = 0
    best_dice_oc = 0
    
    all_logits = []
    all_gts = []
    
    print("Mencari threshold optimal untuk OD dan OC secara terpisah...")
    with torch.no_grad():
        for imgs, seg_gts in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            all_logits.append(logits.cpu())
            all_gts.append(seg_gts.cpu())
            
    all_logits = torch.cat(all_logits)
    all_gts = torch.cat(all_gts)
    
    for t in thresholds:
        dice_od = compute_dice_coef((all_logits[:, 0, :, :] >= t).int(), all_gts[:, 0, :, :].int())
        if dice_od > best_dice_od:
            best_dice_od = dice_od
            best_t_od = t
            
    for t in thresholds:
        dice_oc = compute_dice_coef((all_logits[:, 1, :, :] >= t).int(), all_gts[:, 1, :, :].int())
        if dice_oc > best_dice_oc:
            best_dice_oc = dice_oc
            best_t_oc = t
            
    print(f"Best Threshold OD: {best_t_od:.2f} (Dice: {best_dice_od:.4f})")
    print(f"Best Threshold OC: {best_t_oc:.2f} (Dice: {best_dice_oc:.4f})")
    
    return best_t_od, best_t_oc

best_t_od, best_t_oc = find_best_thresholds_separate(model, test_loader, device)

# **Testing Phase**

In [ ]:
print("\n" + "="*50)
print(f"Wait... Evaluating on TEST SET with OD_t: {best_t_od:.2f}, OC_t: {best_t_oc:.2f}")

model.load_state_dict(torch.load('/kaggle/working/best_seg.pth'))
model.eval()

test_loss = 0.
test_dsc_od = 0.
test_dsc_oc = 0.
test_vCDR_error = 0.
test_vCDRs = []

test_iou_od, test_prec_od, test_rec_od, test_f1_od = 0., 0., 0., 0.
test_iou_oc, test_prec_oc, test_rec_oc, test_f1_oc = 0., 0., 0., 0.

with torch.no_grad():
    for batch_idx, (imgs, seg_gts) in enumerate(test_loader):
        imgs, seg_gts = imgs.to(device), seg_gts.to(device)
        
        logits = model(imgs)
        loss = seg_loss(logits, seg_gts)
        test_loss += loss.item()

        pred_od = (logits[:, 0, :, :] >= best_t_od).type(torch.int8)
        pred_oc = (logits[:, 1, :, :] >= best_t_oc).type(torch.int8)
        gt_od = seg_gts[:, 0, :, :].type(torch.int8)
        gt_oc = seg_gts[:, 1, :, :].type(torch.int8)

        iou_d, prec_d, rec_d, f1_d = compute_all_metrics(pred_od, gt_od)
        test_iou_od += iou_d; test_prec_od += prec_d; test_rec_od += rec_d; test_f1_od += f1_d
        test_dsc_od += compute_dice_coef(pred_od, gt_od).item()

        iou_c, prec_c, rec_c, f1_c = compute_all_metrics(pred_oc, gt_oc)
        test_iou_oc += iou_c; test_prec_oc += prec_c; test_rec_oc += rec_c; test_f1_oc += f1_c
        test_dsc_oc += compute_dice_coef(pred_oc, gt_oc).item()

        v_err, p_vcdr, g_vcdr = compute_vCDR_error(
            pred_od.cpu().numpy(), pred_oc.cpu().numpy(), 
            gt_od.cpu().numpy(), gt_oc.cpu().numpy()
        )
        test_vCDRs += p_vcdr.tolist()
        test_vCDR_error += v_err
        
        print(f'  Test Batch {batch_idx+1}/{len(test_loader)}', end='\r')

num_batches = len(test_loader)

final_avg_loss = test_loss / num_batches
final_avg_dsc_od = test_dsc_od / num_batches
final_avg_dsc_oc = test_dsc_oc / num_batches
final_avg_vcdr_err = test_vCDR_error / num_batches

print('\n' + '='*25 + ' FINAL TEST RESULTS ' + '='*25)
print(f'{"Metric":<15} | {"Optic Disc (OD)":<18} | {"Optic Cup (OC)":<18}')
print("-" * 60)

print(f'{"Dice Score":<15} | {final_avg_dsc_od:<18.4f} | {final_avg_dsc_oc:<18.4f}')
print(f'{"IoU":<15} | {test_iou_od/num_batches:<18.4f} | {test_iou_oc/num_batches:<18.4f}')
print(f'{"Precision":<15} | {test_prec_od/num_batches:<18.4f} | {test_prec_oc/num_batches:<18.4f}')
print(f'{"Recall":<15} | {test_rec_od/num_batches:<18.4f} | {test_rec_oc/num_batches:<18.4f}')
print(f'{"F1-Score":<15} | {test_f1_od/num_batches:<18.4f} | {test_f1_oc/num_batches:<18.4f}')

print("-" * 60)
print(f'Test Loss     : {final_avg_loss:.4f}')
print(f'Test vCDR Err : {final_avg_vcdr_err:.4f}')
print('=' * 60)

In [ ]:
num_batches = len(test_loader)

final_avg_dsc_od = test_dsc_od / num_batches
final_avg_dsc_oc = test_dsc_oc / num_batches

mean_dice = (final_avg_dsc_od + final_avg_dsc_oc) / 2
mean_iou  = (test_iou_od + test_iou_oc) / (2 * num_batches)
mean_prec = (test_prec_od + test_prec_oc) / (2 * num_batches)
mean_rec  = (test_rec_od + test_rec_oc) / (2 * num_batches)
mean_f1   = (test_f1_od + test_f1_oc) / (2 * num_batches)

print('\n' + '='*15 + ' FINAL TEST RESULTS (MEAN OD & OC) ' + '='*15)
print(f'{"Metric":<20} | {"Mean Score":<18}')
print("-" * 45)

print(f'{"Mean Dice Score":<20} | {mean_dice:<18.4f}')
print(f'{"Mean IoU":<20} | {mean_iou:<18.4f}')
print(f'{"Mean Precision":<20} | {mean_prec:<18.4f}')
print(f'{"Mean Recall":<20} | {mean_rec:<18.4f}')
print(f'{"Mean F1-Score":<20} | {mean_f1:<18.4f}')

print("-" * 45)
print(f'{"Test vCDR Error":<20} | {test_vCDR_error/num_batches:<18.4f}')
print('=' * 45)

# **Output Visulization**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def visualize_as_grid(test_loader, model, device, best_t_od, best_t_oc, num_samples=5):
    model.eval()
    
    dataiter = iter(test_loader)
    images, masks = next(dataiter)

    num_samples = min(num_samples, images.shape[0])
    
    images_subset = images[:num_samples]
    masks_subset = masks[:num_samples]

    images_dev = images_subset.to(device)
    with torch.no_grad():
        logits = model(images_dev)
        
        pred_od_all = (logits[:, 0, :, :] >= best_t_od).float().cpu()
        pred_oc_all = (logits[:, 1, :, :] >= best_t_oc).float().cpu()

    fig, axes = plt.subplots(3, num_samples, figsize=(4 * num_samples, 12))
    
    row_labels = ['Fundus Images', 'Ground Truth', 'Baseline (Prediction)']
    
    for row in range(3):
        for col in range(num_samples):
            ax = axes[row, col]
            
            if row == 0:
                img_show = images_subset[col].permute(1, 2, 0).cpu().numpy()
                img_show = np.clip(img_show, 0, 1) # Range [0, 1]
                ax.imshow(img_show)
                if col == 0: ax.set_ylabel('Fundus Images', fontsize=16, fontweight='bold')
                ax.axis('off')
                
            elif row == 1:
                gt_od = masks_subset[col, 0].cpu()
                gt_oc = masks_subset[col, 1].cpu()
                
                combined_gt = 0.5 * gt_od + 0.5 * gt_oc
                
                ax.imshow(combined_gt, cmap='gray', vmin=0, vmax=1)
                if col == 0: ax.set_ylabel('Ground Truth', fontsize=16, fontweight='bold')
                ax.axis('off')

            elif row == 2:
                p_od = pred_od_all[col]
                p_oc = pred_oc_all[col]
                
                combined_pred = 0.5 * p_od + 0.5 * p_oc
                
                ax.imshow(combined_pred, cmap='gray', vmin=0, vmax=1)
                if col == 0: ax.set_ylabel('Baseline\n(Prediction)', fontsize=16, fontweight='bold')
                ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_as_grid(test_loader, model, device, best_t_od, best_t_oc, num_samples=5)

In [ ]:
def visualize_comparison(test_loader, model, device, best_t_od, best_t_oc, num_samples=2, alpha=0.8):
    model.eval()
    
    dataiter = iter(test_loader)
    images, masks = next(dataiter)

    images_dev = images.to(device)
    with torch.no_grad():
        logits = model(images_dev)
        pred_od_all = (logits[:, 0, :, :] >= best_t_od).cpu().numpy()
        pred_oc_all = (logits[:, 1, :, :] >= best_t_oc).cpu().numpy()

    num_samples = min(num_samples, images.shape[0])

    for idx in range(num_samples):
        img_show = images[idx].permute(1, 2, 0).cpu().numpy()
        img_show = np.clip(img_show, 0, 1)
        img_cv = (img_show * 255).astype(np.uint8)
        
        def create_solid_overlay(img, od_mask, oc_mask, alpha_val):
            h, w, _ = img.shape
            color_canvas = np.zeros((h, w, 3), dtype=np.uint8)
            color_canvas[od_mask == 1] = [128, 128, 128] # OD: Gray
            color_canvas[oc_mask == 1] = [0, 0, 0]       # OC: Black
            
            res = img.copy()
            mask_roi = od_mask == 1
            if np.any(mask_roi):
                res[mask_roi] = cv2.addWeighted(img[mask_roi], 1 - alpha_val, 
                                               color_canvas[mask_roi], alpha_val, 0)
            return res

        gt_od = masks[idx, 0, :, :].cpu().numpy()
        gt_oc = masks[idx, 1, :, :].cpu().numpy()
        gt_overlay = create_solid_overlay(img_cv, gt_od, gt_oc, alpha)

        p_od = pred_od_all[idx]
        p_oc = pred_oc_all[idx]
        pred_overlay = create_solid_overlay(img_cv, p_od, p_oc, alpha)

        plt.figure(figsize=(15, 7))
        
        plt.subplot(1, 3, 1)
        plt.title(f"Sample {idx+1}: Original")
        plt.imshow(img_show)
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title("Ground Truth Overlay")
        plt.imshow(gt_overlay)
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.title("Prediction Overlay")
        plt.imshow(pred_overlay)
        plt.axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
visualize_comparison(test_loader, model, device, best_t_od, best_t_oc, num_samples=5)